# Workflow Usage Demonstration

This notebook demonstrates three different ways to use the same workflow functions:
1. **Direct Python** - Framework-agnostic, pure Python usage
2. **Pyiron** - Using pyiron_workflow decorators with knowledge graph support
3. **Jobflow** - Using jobflow decorators for workflow management

In [1]:
import matplotlib.pyplot as plt
import numpy as np

## Setup: Common Parameters

These parameters will be used across all three approaches.

In [2]:
# Common simulation parameters
pair_style = "eam/alloy"
pair_coeff = "* * workflows/potentials/Fe_Ack.eam Fe"

## 1. Direct Python Usage

Using the core workflow functions directly without any framework decorators.
This is useful for simple scripts, testing, or when you don't need workflow management.

In [3]:
from workflows.evcurves import calculate_ev_curves as calc_ev_python
from workflows.build import bulk as bulk_python

# Create structure using pure Python
structure = bulk_python('Fe', crystalstructure='bcc', a=2.87, cubic=True)

# Calculate EV curves directly
results = calc_ev_python(
    structure, 
    pair_style, 
    pair_coeff, 
    vol_range=0.01,
    num_of_points=5,
    cores=1
)

print("Python Results:")
print(f"  Bulk modulus: {results['bulk_modulus']:.2f} GPa")
print(f"  Volume range: {results['volume'].min():.3f} - {results['volume'].max():.3f} Å³")

ModuleNotFoundError: No module named 'workflows.build'

In [ ]:
# Plot results
plt.figure(figsize=(8, 5))
plt.plot(results['volume'], results['energy'], 'o-', label='Direct Python')
plt.xlabel('Volume (Å³/atom)')
plt.ylabel('Energy (eV/atom)')
plt.title('EV Curve - Direct Python')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Pyiron Workflow Usage

Using pyiron_workflow decorators with knowledge graph support for metadata tracking.
This enables workflow visualization, execution tracking, and semantic metadata storage.

In [ ]:
from workflows.pyiron.build import bulk as bulk_pyiron
from workflows.pyiron.evcurves import calculate_ev_curves as calc_ev_pyiron
from pyiron_workflow import Workflow
from conceptual_dictionary import ConceptualDict 

# Initialize knowledge graph
cd = ConceptualDict()

# Create workflow
wf = Workflow('ev_demo')

# Build structure with KG tracking
wf.structure = bulk_pyiron('Fe', crystalstructure='bcc', a=2.87, cubic=True, kg=cd)

# Calculate EV curves with KG tracking
wf.ev_curves = calc_ev_pyiron(
    wf.structure, 
    pair_style, 
    pair_coeff, 
    vol_range=0.01,
    num_of_points=5,
    cores=1,
    kg=cd,
    potential_type='EAM',
    potential_doi='https://doi.org/10.1103/PhysRevB.69.144113'
)

# Execute workflow
pyiron_results = wf.run()

print("\nPyiron Results:")
print(f"  Bulk modulus: {pyiron_results['ev_curves__datadict']['bulk_modulus']:.2f} GPa")
print(f"  Workflow nodes: {list(wf.nodes.keys())}")
print(f"  Knowledge graph samples: {len(cd['computational_sample'])}")
print(f"  Knowledge graph workflows: {len(cd['workflow'])}")

In [ ]:
# Plot results
plt.figure(figsize=(8, 5))
plt.plot(
    pyiron_results['ev_curves__datadict']['volume'], 
    pyiron_results['ev_curves__datadict']['energy'], 
    's-', 
    label='Pyiron Workflow'
)
plt.xlabel('Volume (Å³/atom)')
plt.ylabel('Energy (eV/atom)')
plt.title('EV Curve - Pyiron Workflow')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualize workflow graph
wf.draw(rankdir='LR')

In [ ]:
# Export knowledge graph
cd.to_yaml('demo_evcurve.yaml')
print("Knowledge graph exported to demo_evcurve.yaml")

## 3. Jobflow Usage

Using jobflow decorators for distributed workflow execution and job management.
This enables job submission to queuing systems and distributed computing.

In [ ]:
from workflows.jobflow.build import bulk as bulk_jobflow
from workflows.jobflow.evcurves import calculate_ev_curves as calc_ev_jobflow
from jobflow import Flow, run_locally

# Create structure job
structure_job = bulk_jobflow('Fe', crystalstructure='bcc', a=2.87, cubic=True)

# Create EV curves job (depends on structure job)
ev_job = calc_ev_jobflow(
    structure_job.output,  # Use output from previous job
    pair_style,
    pair_coeff,
    vol_range=0.01,
    num_of_points=5,
    cores=1
)

# Create flow
flow = Flow([structure_job, ev_job], name='ev_demo_flow')

# Run flow locally
jobflow_results = run_locally(flow)

# Extract results
final_output = jobflow_results[ev_job.uuid][1].output

print("\nJobflow Results:")
print(f"  Bulk modulus: {final_output['bulk_modulus']:.2f} GPa")
print(f"  Jobs executed: {len(jobflow_results)}")

In [ ]:
# Plot results
plt.figure(figsize=(8, 5))
plt.plot(
    final_output['volume'], 
    final_output['energy'], 
    '^-', 
    label='Jobflow'
)
plt.xlabel('Volume (Å³/atom)')
plt.ylabel('Energy (eV/atom)')
plt.title('EV Curve - Jobflow')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Comparison: All Three Approaches

Let's compare the results from all three approaches side by side.

In [ ]:
# Compare all results
plt.figure(figsize=(10, 6))

plt.plot(results['volume'], results['energy'], 'o-', label='Direct Python', alpha=0.7)
plt.plot(pyiron_results['ev_curves__datadict']['volume'], 
         pyiron_results['ev_curves__datadict']['energy'], 
         's-', label='Pyiron', alpha=0.7)
plt.plot(final_output['volume'], final_output['energy'], 
         '^-', label='Jobflow', alpha=0.7)

plt.xlabel('Volume (Å³/atom)', fontsize=12)
plt.ylabel('Energy (eV/atom)', fontsize=12)
plt.title('EV Curves - All Approaches Comparison', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nBulk Modulus Comparison:")
print(f"  Direct Python: {results['bulk_modulus']:.2f} GPa")
print(f"  Pyiron:        {pyiron_results['ev_curves__datadict']['bulk_modulus']:.2f} GPa")
print(f"  Jobflow:       {final_output['bulk_modulus']:.2f} GPa")

## Summary

### Direct Python
- ✅ Simplest approach, no framework overhead
- ✅ Easy to integrate into existing scripts
- ❌ No workflow tracking or visualization
- ❌ No metadata/provenance capture

### Pyiron Workflow
- ✅ Workflow visualization and execution graph
- ✅ Knowledge graph integration for semantic metadata
- ✅ Interactive workflow development
- ❌ Requires pyiron_workflow framework

### Jobflow
- ✅ Designed for distributed/HPC execution
- ✅ Job submission to queuing systems
- ✅ Automatic handling of job dependencies
- ❌ Requires jobflow framework setup

**Key Insight:** All three approaches use the same core computational functions from `workflows/`, 
just with different decorators applied. This means you maintain the logic once and can use 
it flexibly depending on your needs!